In [1]:
import concurrent.futures
import dataclasses
import io

import fsspec
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pacer.external.speedhive import SpeedhiveSession, parse_laptime_seconds

# Where's the quali?

There's no actual qualies in the [speedhive](https://speedhive.mylaps.com/events/3660570). Is it the case of not attached id (in between practice and race) ones?

In [ ]:
def _possible(id_: int) -> bool:
    try:
        session = SpeedhiveSession(session_id=id_, name="2026-aug-mk-3h-quali")
        results_df = session.results_df()
    except:
        return False
    return (
        "Competitor" in results_df.columns
        and (results_df["Competitor"] == "James King").any()
    )


ids = range(12644500, 12648077)

with concurrent.futures.ThreadPoolExecutor(16) as executor:
    mask = executor.map(possible, ids)

mask = list(mask)
pd.Series(mask, index=ids).loc[lambda s: s]

# Race analysis

In [32]:
mk_3hr = SpeedhiveSession(session_id=12648075, name="2026-jul-sp-3h")
laptimes = mk_3hr.laptimes()
results_df = mk_3hr.results_df()
results_df

,Start Number,Competitor,Class,Total Time,Diff,Laps,Best Lap,Best Lap No.,Best Speed
1,141,DHC Racing,2026 Endurance 2 Stroke,0 days 03:00:33.073000,0.000,149,1:5.780,15,74.430 km/h
2,134,James King,2026 Endurance 2 Stroke,0 days 03:01:38.896000,1:5.823,149,1:6.078,18,74.094 km/h
3,130,SD Kart,2026 Endurance 2 Stroke,0 days 03:01:34.975000,1 lap,148,1:6.671,23,73.435 km/h
4,146,DNH-Max Iron Dames,2026 Endurance 2 Stroke,0 days 03:00:45.881000,2 laps,147,1:7.029,28,73.043 km/h
5,147,The Mandem - Tom Avoh,2026 Endurance 2 Stroke,0 days 03:00:59.180000,2 laps,147,1:6.014,10,74.166 km/h
6,150,Clean Bulls Racing,2026 Endurance 2 Stroke,0 days 03:01:44.873000,2 laps,147,1:6.297,2,73.849 km/h
7,152,Maple Motorsport,2026 Endurance 2 Stroke,0 days 03:00:37.244000,3 laps,146,1:5.994,7,74.189 km/h
8,157,AMC,2026 Endurance 2 Stroke,0 days 02:59:27.766000,4 laps,145,1:6.989,108,73.087 km/h
9,149,WPR,2026 Endurance 2 Stroke,0 days 03:00:55.419000,4 laps,145,1:6.981,24,73.095 km/h
10,148,DNH-Max Hill's Heroes,2026 Endurance 2 Stroke,0 days 03:00:37.669000,5 laps,144,1:6.729,10,73.371 km/h


In [33]:
start_times = results_df.set_index("Competitor")["Total Time"] - laptimes.sum(axis=0)

In [34]:
timestamps = laptimes.assign(
    **{c: lambda d, c=c: d[c].cumsum() + start_times[c] for c in laptimes.columns}
)

In [72]:
def plot_laptime_vs_timestamp(
    timestamps: pd.DataFrame, laptimes: pd.DataFrame, results_df: pd.DataFrame
) -> go.Figure:
    ts = (timestamps.stack().dt.total_seconds() / 60).rename("timestamp_m")
    lt = laptimes.stack().dt.total_seconds().rename("laptime_s")
    df = pd.concat([ts, lt], axis=1).reset_index()
    df.columns = ["lap", "competitor", "timestamp_m", "laptime_s"]

    competitor_to_class = results_df.set_index("Competitor")["Class"]
    df["class"] = df["competitor"].map(competitor_to_class)

    fig = px.scatter(
        df,
        x="timestamp_m",
        y="laptime_s",
        color="class",
        hover_data=["lap", "competitor"],
        log_y=True,
        title="Lap time vs race time - 2026 Jul Daytona SP 3h",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.update_layout(
        xaxis_title="Race time (m)",
        yaxis_title="Lap time (s)",
        legend_title="Class",
    )
    return fig


plot_laptime_vs_timestamp(timestamps, laptimes, results_df)

In [36]:
RACE_STATES = ["green flag lap", "safety kart lap", "pitstop", "safety kart pitstop"]
LAP_STATES = RACE_STATES + ["timing glitch"]


def classify_laps(
    timestamps: pd.DataFrame,
    laptimes: pd.DataFrame,
    *,
    green_quantile: float = 0.3,
    safety_ratio: float = 1.2,
    field_window: str = "150s",
    pit_excess_s: float = 30.0,
    glitch_ratio: float = 0.85,
) -> pd.DataFrame:
    """Label every lap green flag / safety kart / pitstop / safety kart pitstop.

    A safety kart period is a property of the race, a pitstop is a property of one
    competitor: the whole field slows together for the kart, but only a handful of
    teams are stopped at any moment.  The two are therefore detected on different
    signals rather than on lap time alone.

    Each competitor gets their own green reference pace (`green_quantile` of their
    laps), which puts DMAX and SODI on one scale.  The *field median* of
    lap/reference over a rolling `field_window` of race time is then a pace
    multiplier for the race as a whole - flat near 1.0 under green, well above it
    when the kart is out - and pitstops cannot move it, because a median ignores
    the few teams stopped at a time.  Whatever a competitor loses on top of that
    shared multiplier is their own time loss, and more than `pit_excess_s` of it
    is a stop.

    Laps below `glitch_ratio` of the competitor's own reference are impossible to
    drive and are labelled `timing glitch` (a transponder counting one crossing
    twice splits a real lap into fragments), so they are not mistaken for
    exceptionally quick green laps.
    """
    lap_end = timestamps.stack().dt.total_seconds().rename("lap_end_s")
    lap_s = laptimes.stack().dt.total_seconds().rename("laptime_s")
    laps = (
        pd.concat([lap_end, lap_s], axis=1)
        .dropna()
        .rename_axis(index=["lap", "competitor"])
        .reset_index()
        .sort_values("lap_end_s", ignore_index=True)
    )

    green_ref = laptimes.apply(lambda s: s.dt.total_seconds()).quantile(green_quantile)
    laps["green_ref_s"] = laps["competitor"].map(green_ref)
    laps["ratio"] = laps["laptime_s"] / laps["green_ref_s"]

    laps["field_ratio"] = (
        laps["ratio"]
        .set_axis(pd.to_timedelta(laps["lap_end_s"], unit="s"))
        .rolling(field_window, center=True)
        .median()
        .to_numpy()
    )
    laps["expected_s"] = laps["green_ref_s"] * laps["field_ratio"].clip(lower=1.0)
    laps["excess_s"] = laps["laptime_s"] - laps["expected_s"]

    laps["safety"] = laps["field_ratio"] > safety_ratio
    laps["pit"] = laps["excess_s"] > pit_excess_s
    laps["glitch"] = laps["ratio"] < glitch_ratio
    laps["state"] = pd.Categorical(
        np.where(
            laps["glitch"],
            "timing glitch",
            np.where(
                laps["pit"],
                np.where(laps["safety"], "safety kart pitstop", "pitstop"),
                np.where(laps["safety"], "safety kart lap", "green flag lap"),
            ),
        ),
        categories=LAP_STATES,
        ordered=True,
    )
    return laps


def lap_state_summary(laps: pd.DataFrame) -> pd.DataFrame:
    """Table twin of the scatter: how many laps per bucket, and how slow they run."""
    return (
        laps.groupby("state", observed=False)["laptime_s"]
        .agg(laps="size", fastest="min", median="median", slowest="max")
        .assign(share=lambda d: (d["laps"] / d["laps"].sum()).map("{:.1%}".format))
        .round(1)
    )


laps = classify_laps(timestamps, laptimes)
lap_state_summary(laps)

,laps,fastest,median,slowest,share
state,,,,,
green flag lap,6671,65.7,75.5,116.5,92.5%
safety kart lap,206,69.1,129.9,167.6,2.9%
pitstop,291,108.4,179.7,834.1,4.0%
safety kart pitstop,42,146.3,255.1,392.0,0.6%
timing glitch,0,NaN,NaN,NaN,0.0%


In [37]:
laps

,lap,competitor,lap_end_s,laptime_s,green_ref_s,ratio,field_ratio,expected_s,excess_s,safety,pit,glitch,state
0,0,Triple Threat,72.219,68.242,66.9180,1.019785,1.036749,69.377157,-1.135157,False,False,False,green flag lap
1,0,James King,73.253,68.673,67.0316,1.024487,1.035011,69.378460,-0.705460,False,False,False,green flag lap
2,0,Maple Motorsport,73.587,68.976,67.4445,1.022708,1.035011,69.805816,-0.829816,False,False,False,green flag lap
3,0,The Mandem - Tom Avoh,74.374,69.517,67.2784,1.033274,1.032413,69.459065,0.057935,False,False,False,green flag lap
4,0,TSB Motorsport,74.847,69.522,68.3127,1.017702,1.032413,70.526889,-1.004889,False,False,False,green flag lap
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7205,124,Iron Dames by DNH,10905.919,81.192,79.5400,1.020769,1.019650,81.102942,0.089058,False,False,False,green flag lap
7206,125,Ickx's Icons by DNH,10910.552,83.678,77.0145,1.086523,1.023663,78.836879,4.841121,False,False,False,green flag lap
7207,134,Miles-Shelby Squad by DNH,10910.724,83.580,74.8798,1.116189,1.023663,76.651666,6.928334,False,False,False,green flag lap
7208,131,J J J Racing,10911.571,79.352,72.6551,1.092174,1.023663,74.374324,4.977676,False,False,False,green flag lap


In [38]:
# Lap-time ticks at readable motorsport intervals - a bare log axis renders "5" for
# 50s and again for 500s, which cannot be read.
_LAPTIME_TICKS = [45, 60, 90, 120, 180, 300, 600]
_LAPTIME_TICKTEXT = ["45s", "1:00", "1:30", "2:00", "3:00", "5:00", "10:00"]


def plot_lap_states(laps: pd.DataFrame, title: str | None = None) -> go.Figure:
    """Lap time vs race time, coloured by flag condition, shaped by pit / racing.

    Four buckets on two hues: colour carries the race-wide flag condition and
    marker shape the individual stop, so the eye never has to separate four
    colours at once.  Timing glitches are left out - they are not a race state,
    and a 14s lap stretches a log axis over a decade that holds no data - but they
    stay in `laps` and are counted in `lap_state_summary`.
    """
    df = laps[~laps["glitch"]].assign(
        race_time_min=lambda d: d["lap_end_s"] / 60,
        flag=lambda d: np.where(d["safety"], "safety kart", "green flag"),
        lap_type=lambda d: np.where(d["pit"], "pitstop", "racing"),
    )
    dropped = int(laps["glitch"].sum())
    fig = px.scatter(
        df,
        x="race_time_min",
        y="laptime_s",
        color="flag",
        symbol="lap_type",
        log_y=True,
        category_orders={
            "flag": ["green flag", "safety kart"],
            "lap_type": ["racing", "pitstop"],
        },
        color_discrete_map={"green flag": "#2a78d6", "safety kart": "#eb6834"},
        symbol_map={"racing": "circle", "pitstop": "diamond"},
        hover_name="competitor",
        hover_data={
            "lap": True,
            "state": True,
            "laptime_s": ":.2f",
            "excess_s": ":.1f",
            "race_time_min": ":.1f",
        },
        title=title or "Lap time by race condition",
        subtitle=f"{len(df):,} laps"
        + (f" · {dropped} timing glitches excluded" if dropped else ""),
    )
    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.update_traces(
        marker=dict(size=7, opacity=0.9), selector=lambda t: "pitstop" in t.name
    )
    fig.update_layout(
        template="plotly_white",
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="#fcfcfb",
        font=dict(color="#52514e"),
        title=dict(font=dict(color="#0b0b0b")),
        legend=dict(title_text="", itemsizing="constant"),
        margin=dict(t=80, r=20),
    )
    fig.update_xaxes(title="Race time (min)", gridcolor="#e1e0d9", zeroline=False)
    fig.update_yaxes(
        title="Lap time",
        gridcolor="#e1e0d9",
        zeroline=False,
        tickmode="array",
        tickvals=_LAPTIME_TICKS,
        ticktext=_LAPTIME_TICKTEXT,
    )
    return fig


plot_lap_states(
    laps.loc[lambda d: d["competitor"].isin(laptimes.columns.tolist()[:4])],
    "Lap time by race condition — 2026 Jul Daytona SP 3h",
)

In [39]:
px.bar(start_times.dt.total_seconds())

In [40]:
mk_3hr.competitor_data(1)

,Lap,Pos,Lap Time,Diff to Last Lap,Diff to Best Lap,Gap in Front,Diff to P1,Speed
0,1,16,0 days 00:01:11.514000,0.000,5.734,1.568,10.479,68.462 km/h
1,2,16,0 days 00:01:06.473000,-,0.693,0.043,10.908,73.654 km/h
2,3,14,0 days 00:01:06.241000,-,0.461,0.904,9.818,73.912 km/h
3,4,13,0 days 00:01:06.893000,0.652,1.113,0.615,10.641,73.192 km/h
4,5,13,0 days 00:01:07.971000,1.078,2.191,0.158,7.414,72.031 km/h
...,...,...,...,...,...,...,...,...
144,145,1,0 days 00:01:07.478000,-,1.698,0.000,0.000,72.557 km/h
145,146,1,0 days 00:01:07.126000,-,1.346,0.000,0.000,72.937 km/h
146,147,1,0 days 00:01:07.368000,0.242,1.588,0.000,0.000,72.675 km/h
147,148,1,0 days 00:01:06.857000,-,1.077,0.000,0.000,73.231 km/h


In [70]:
def plot_gap_to_mean_leader(laptimes: pd.DataFrame) -> go.Figure:
    leader_mean_lap = laptimes.iloc[:, 0].mean()
    return px.line(
        laptimes.assign(
            **{
                col: lambda d, col=col: (
                    (leader_mean_lap - d[col]).cumsum().dt.total_seconds()
                )
                for col in laptimes.columns
            }
        ),
        title="Gap to leader mean lap time (cumulative) - 2026 Jul Dayona SP 3h",
    )


plot_gap_to_mean_leader(laptimes)

In [45]:
timestamps

,DHC Racing,James King,SD Kart,DNH-Max Iron Dames,The Mandem - Tom Avoh,Clean Bulls Racing,Maple Motorsport,AMC,WPR,DNH-Max Hill's Heroes,...,Skill Issue Motorsports,Gran Autismo,Team Maverick,sBinalla Racing,BAT Motorsport,Two Speed Racing,Knot Fast Enough,Spin Bandits,MIND,The Mandem
0,0 days 00:01:22.698000,0 days 00:01:13.253000,0 days 00:01:17.030000,0 days 00:01:18.125000,0 days 00:01:14.374000,0 days 00:02:28.636000,0 days 00:01:13.587000,0 days 00:01:16.756000,0 days 00:01:19.792000,0 days 00:01:17.594000,...,0 days 00:01:39.872000,0 days 00:01:40.828000,0 days 00:01:42.642000,0 days 00:01:47.324000,0 days 00:01:37.474000,0 days 00:01:42.025000,0 days 00:01:28.709000,0 days 00:01:49.795000,0 days 00:01:17.454000,0 days 00:01:15.273000
1,0 days 00:02:29.171000,0 days 00:02:19.728000,0 days 00:02:25.378000,0 days 00:02:26.224000,0 days 00:02:20.776000,0 days 00:03:34.933000,0 days 00:02:20.010000,0 days 00:02:25.004000,0 days 00:02:27.115000,0 days 00:02:25.968000,...,0 days 00:02:58.804000,0 days 00:02:59.516000,0 days 00:03:02.617000,0 days 00:03:09.342000,0 days 00:02:56.172000,0 days 00:03:01.970000,0 days 00:02:45.055000,0 days 00:03:14.934000,0 days 00:02:25.660000,0 days 00:02:22.284000
2,0 days 00:03:35.412000,0 days 00:03:26.601000,0 days 00:03:32.487000,0 days 00:03:34.508000,0 days 00:03:27.695000,0 days 00:04:42.097000,0 days 00:03:27.076000,0 days 00:03:32.335000,0 days 00:03:34.133000,0 days 00:03:33.010000,...,0 days 00:04:17.895000,0 days 00:04:17.734000,0 days 00:04:20.202000,0 days 00:04:30.347000,0 days 00:04:13.482000,0 days 00:04:20.898000,0 days 00:04:01.138000,0 days 00:05:44.651000,0 days 00:03:32.815000,0 days 00:03:29.544000
3,0 days 00:04:42.305000,0 days 00:04:32.914000,0 days 00:04:40.525000,0 days 00:04:44.009000,0 days 00:04:34.253000,0 days 00:05:50.178000,0 days 00:04:33.335000,0 days 00:04:39.656000,0 days 00:04:41.690000,0 days 00:04:41.181000,...,0 days 00:05:41.499000,0 days 00:05:41.300000,0 days 00:05:41.843000,0 days 00:05:51.488000,0 days 00:05:35.474000,0 days 00:05:43.166000,0 days 00:05:19.919000,0 days 00:07:14.187000,0 days 00:04:40.637000,0 days 00:04:36.123000
4,0 days 00:05:50.276000,0 days 00:05:43.295000,0 days 00:05:48.197000,0 days 00:05:53.182000,0 days 00:05:43.503000,0 days 00:06:59.206000,0 days 00:05:43.187000,0 days 00:05:47.788000,0 days 00:05:49.834000,0 days 00:05:50.118000,...,0 days 00:07:00.812000,0 days 00:06:59.945000,0 days 00:07:04.758000,0 days 00:07:11.656000,0 days 00:06:51.924000,0 days 00:07:15.047000,0 days 00:06:36.884000,0 days 00:08:42.102000,0 days 00:05:49.022000,0 days 00:05:45.354000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,0 days 02:56:03.864000,0 days 02:56:55.931000,0 days 02:58:00.996000,0 days 02:58:27.954000,0 days 02:58:43.211000,0 days 02:59:20.985000,0 days 02:59:28.604000,0 days 02:59:27.766000,0 days 03:00:55.419000,NaT,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
145,0 days 02:57:10.990000,0 days 02:58:04.337000,0 days 02:59:08.656000,0 days 02:59:37.430000,0 days 02:59:50.611000,0 days 03:00:29.409000,0 days 03:00:37.244000,NaT,NaT,NaT,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
146,0 days 02:58:18.358000,0 days 02:59:12.243000,0 days 03:00:16.754000,0 days 03:00:45.881000,0 days 03:00:59.180000,0 days 03:01:44.873000,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
147,0 days 02:59:25.215000,0 days 03:00:20.693000,0 days 03:01:34.975000,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT


In [69]:
def plot_gap_to_mean_leader(
    timestamps: pd.DataFrame, laptimes: pd.DataFrame
) -> go.Figure:
    leader_mean_lap = laptimes.iloc[:, 0].mean()

    leader_pace_timestamps = pd.Series(
        np.arange(timestamps.shape[0]) * leader_mean_lap, index=timestamps.index
    )

    gap = pd.DataFrame({c: leader_pace_timestamps for c in laptimes.columns}) - timestamps
    gap = gap.assign(**{c: lambda d, c=c: d[c].dt.total_seconds() for c in gap.columns})

    race_time = timestamps.stack().dt.total_seconds() / 60

    df = (
        pd.concat(
            [race_time.rename("race_time_min"), gap.stack().rename("gap_s")], axis=1
        )
        .rename_axis(index=["lap", "competitor"])
        .reset_index()
    )

    fig = px.line(
        df,
        x="race_time_min",
        y="gap_s",
        color="competitor",
        hover_data=["lap"],
        title="Gap to leader mean lap time (cumulative) - 2026 Jul Daytona SP 3h",
    )
    fig.update_layout(
        xaxis_title="Race time (min)", yaxis_title="Gap (s)", legend_title=""
    )
    return fig


plot_gap_to_mean_leader(timestamps, laptimes)


In [73]:
def plot_laptime_distribution(laptimes: pd.DataFrame) -> go.Figure:
    best_lap = laptimes.where(lambda d: d > d.median().min() * 0.93).min().min()
    return (
        laptimes.where(lambda d: (d > best_lap * 0.93) & (d < best_lap * 1.07))
        .assign(
            **{
                col: lambda d, c=col: d[c].dt.total_seconds()
                for col in laptimes.columns
            }
        )
        .pipe(px.violin, box=True)#, points="all")
    )


plot_laptime_distribution(laptimes[lambda d: d.columns[:10]])